In [111]:
import pandas as pd
import os
import numpy as np

# =====================================================================
# ⚙️ 全局配置区
# =====================================================================
EXCEL_FILE = "CS2_Project.xlsx"
SHEET_RAW = "Raw Data"      # 你的倾倒垃圾桶（中转页）
SHEET_CLEAN = "Clean Data"  # 你的终点站（干净大表）

TEAM_CONFIG = {
    'Vitality':{
    'Mirage': {
        'ropz':  {'T': 'A extremity',   'CT': 'B short'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Roamer',  'CT': 'A connector'},
        'flameZ': {'T': 'Mid Brawl',      'CT': 'A anchor'},
        'mezii':  {'T': 'B extremity',      'CT': 'B anchor'}
    },
    'Dust2': {
        'ropz':  {'T': 'Upper B',   'CT': 'B anchor'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Middle',  'CT': 'CT'},
        'flameZ': {'T': 'Lower B',      'CT': 'Long'},
        'mezii':  {'T': 'Long',      'CT': 'Middle'}
    },
    'Inferno': {
        'ropz':  {'T': 'Apps',   'CT': 'Pit'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Roamer',  'CT': 'B support'},
        'flameZ': {'T': 'Middle',      'CT': 'B Aggressor'},
        'mezii':  {'T': 'Banana',      'CT': 'A Rifle'}
    },
    'Nuke': {
        'ropz':  {'T': 'Lobby',   'CT': 'Ramp'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Outside',  'CT': 'Outside'},
        'flameZ': {'T': 'Silo',      'CT': 'Door'},
        'mezii':  {'T': 'Support',      'CT': 'Floor'}
    },
    'Anubis': {
        'ropz':  {'T': 'A extremity',   'CT': 'A anchor'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Canals',  'CT': 'B connector'},
        'flameZ': {'T': 'Middle',      'CT': 'Middle'},
        'mezii':  {'T': 'B extremity',      'CT': 'B anchor'}
    },
    'Train': {
        'ropz':  {'T': 'Unknown',   'CT': 'A Unknown'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Unknown',  'CT': 'Unknown'},
        'flameZ': {'T': 'Unknown',      'CT': 'Unknown'},
        'mezii':  {'T': 'Unknown',      'CT': 'Unknown'}
    },
    'Overpass': {
        'ropz':  {'T': 'B extremity',   'CT': 'A rifle'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Roamer',  'CT': 'B rotate'},
        'flameZ': {'T': 'A Toilets',      'CT': 'B short'},
        'mezii':  {'T': 'Connector',      'CT': 'B monster'}
    },
    'Ancient': {
        'ropz':  {'T': 'A extremity',   'CT': 'A anchor'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'B cave',  'CT': 'B cave'},
        'flameZ': {'T': 'Middle',      'CT': 'Middle'},
        'mezii':  {'T': 'B extremity',      'CT': 'B anchor'}
    }
    },
    'FURIA':{
        'Mirage':{
            'YEKINDAR': {'T': 'Mid Brawl', 'CT': 'B short'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'Roamer', 'CT': 'B anchor'},
            'KSCERATO': {'T': 'B extremity', 'CT': 'A connector'},
            'yuurih': {'T': 'A extremity', 'CT': 'A anchor'}
        }
    }
}
print("✅ 配置加载成功")

def clean_cs2_side_table(team, map_name, opponent="", stage="", match_name=""):
    if team not in TEAM_CONFIG:
        print(f"❌ 未找到队伍 {team}")
        return
    if map_name not in TEAM_CONFIG[team]:
        print(f"❌ 错误：未找到地图 【{map_name}】")
        return
    if not os.path.exists(EXCEL_FILE):
        print(f"❌ 错误：未找到文件 【{EXCEL_FILE}】")
        return

    df_raw = pd.read_excel(EXCEL_FILE, sheet_name="Staging")
    if df_raw.empty:
        print("ℹ️ 【Staging】 影子表是空的。")
        return

    def normalize_player(name_val):
        name_str = str(name_val).strip().lower()
        for standard_name in TEAM_CONFIG[team][map_name].keys():
            if standard_name.lower() in name_str:
                return standard_name
        return 'Unknown'

    df_t = df_raw.iloc[:, [0, 5, 3, 2]].copy()
    df_t.columns = ['Raw_Player', 'T-side Rating', 'T-side ADR', 'Raw_Swing']
    df_t['Player'] = df_t['Raw_Player'].apply(normalize_player)
    df_t = df_t[df_t['Player'] != 'Unknown'].copy()
    df_t['T-side Round Swing'] = df_t['Raw_Swing'].astype(float)
    df_t['T-side Position'] = df_t['Player'].map(lambda p: TEAM_CONFIG[team][map_name][p]['T'])
    df_t = df_t[['Player', 'T-side Position', 'T-side Rating', 'T-side ADR', 'T-side Round Swing']]

    df_ct = df_raw.iloc[:, [7, 12, 10, 9]].copy()
    df_ct.columns = ['Raw_Player', 'CT-side Rating', 'CT-side ADR', 'Raw_Swing']
    df_ct['Player'] = df_ct['Raw_Player'].apply(normalize_player)
    df_ct = df_ct[df_ct['Player'] != 'Unknown'].copy()
    df_ct['CT-side Round Swing'] = df_ct['Raw_Swing'].astype(float)
    df_ct['CT-side Position'] = df_ct['Player'].map(lambda p: TEAM_CONFIG[team][map_name][p]['CT'])
    df_ct = df_ct[['Player', 'CT-side Position', 'CT-side Rating', 'CT-side ADR', 'CT-side Round Swing']]

    df_valid = pd.merge(df_t, df_ct, on='Player', how='inner')
    
    if df_valid.empty:
        print("⚠️ 警告：没有识别到有效队员，运行终止。")
        return None
    if len(df_valid) != 5:
        print(f"⚠️ 本场比赛识别到了 {len(df_valid)} 名选手，请检查。")
        return
    
    # 读取历史大表（Clean Data）
    try:
        df_old_master = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_CLEAN)
    except Exception:
        print("❌ 错误：无法读取到历史大表。")
        return

    # 智能检查并补齐大表的维度
    # target_columns = list(df_old_master.columns)
    # if 'Stage' not in target_columns:
    #    target_columns.insert(5, 'Stage')
    #    df_old_master['Stage'] = "Historical Stage"

    # 构建内存新表
    #    彻底解决：赛事名吃你手动传入的值，队伍名锁定为 Vitality
    df_new = pd.DataFrame({
       "Match": match_name,
       "Team": team,
       "Player": df_valid['Player'].values,
       "Map": map_name,
       "Opponent": opponent,
       "Stage": stage,

        "T-side Position": df_valid["T-side Position"].values,
        "CT-side Position": df_valid["CT-side Position"].values,

        "T-side Rating": df_valid["T-side Rating"].values,
        "CT-side Rating": df_valid["CT-side Rating"].values,

        "T-side ADR": df_valid["T-side ADR"].values,
        "CT-side ADR": df_valid["CT-side ADR"].values,

        "T-side Round Swing": df_valid["T-side Round Swing"].values,
        "CT-side Round Swing": df_valid["CT-side Round Swing"].values,

        "First Kill": np.nan,
        "Multi-kill Rating": np.nan
    })
    # df_new = pd.DataFrame(None, columns=target_columns)
    # df_new['Match'] = match_name * len(df_valid)
    # df_new['Team'] = TEAM_NAME * len(df_valid)
    # df_new['Player'] = df_valid['Player'].values
    # df_new['Map'] = map_name
    # df_new['Opponent'] = opponent
    # df_new['Stage'] = stage

    print(df_new[
        [
            "Match",
            "Team",
            "Player",
            "Map"
        ]
    ])

    df_updated_master = pd.concat(
        [df_old_master, df_new],
        ignore_index=True
    )

    # 5. 安全覆盖写回 Excel（记得先关闭你的 Excel 软件！）
    with pd.ExcelWriter(EXCEL_FILE, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_updated_master.to_excel(writer, sheet_name=SHEET_CLEAN, index=False)
        
    print(f"🎉 成功！本次洗好的 【{map_name}】地图数据已安全塞入大表！")
    return df_updated_master

print("✅ 清洗与追加引擎加载成功！")


# =====================================================================
# 🎮 日常操作台：每次在这里下达你要清洗的是哪张图、哪个半场
# =====================================================================

# Maps: Dust2, Mirage, Inferno, Ancient, Nuke, Anubis, Train, Overpass
# Teams: Vitality, Falcons, Liquid, MOUZ, Spirit, FaZe, Natus Vincere, The MongolZ, G2, GamerLegion, Virtus.pro, Astralis, FURIA, Legacy, Aurora, PARIVISION, FUT, 
# Stages: Group stage double elimination, Single-elimination, Upper bracket final, Lower bracket decider, Playoff, Grand final, Swiss Round, Major stage 1 Swiss round, Major stage 2 Swiss round, Major stage 3 Swiss round, Major playoff, Major grand final, 3rd place decider
# Stage System 1: Swiss rounds (Swiss Round)
# Stage System 2: IEM (Group stage double elimination, Upper bracket final / Qualification match, Lower bracket decider)
# Stage System 3: BLAST (Group stage, Qualification Match， Elimination Match)
# Stage System 4: All single elimination (Single-elimination)
# Stage System 5: Major (Major stage 1 Swiss round, Major stage 2 Swiss round, Major stage 3 Swiss round, Major playoff, Major grand final)
# Shared stages: Playoff, Grand final, 3rd place decider

# 假设你刚刚在 Excel 的 Raw_Staging 里贴了 Mirage 的 T 侧表格：
current_analysis = clean_cs2_side_table(
    team="FURIA",
    map_name="Mirage", 
    opponent="The MongolZ", 
    stage="Swiss Round",
    match_name="PGL Astana 2025"
)

# 预览这次处理完后的大表全貌
current_analysis.tail(10)

✅ 配置加载成功
✅ 清洗与追加引擎加载成功！
             Match   Team    Player     Map
0  PGL Astana 2025  FURIA  KSCERATO  Mirage
1  PGL Astana 2025  FURIA    FalleN  Mirage
2  PGL Astana 2025  FURIA  YEKINDAR  Mirage
3  PGL Astana 2025  FURIA   molodoy  Mirage
4  PGL Astana 2025  FURIA    yuurih  Mirage
🎉 成功！本次洗好的 【Mirage】地图数据已安全塞入大表！


,Match,Stage,Team,Opponent,Player,Map,T-side Position,CT-side Position,T-side Rating,CT-side Rating,T-side ADR,CT-side ADR,T-side Round Swing,CT-side Round Swing,First Kill,Multi-kill Rating
1490,IEM Cologne Major 2026,Major playoff,Vitality,Falcons,ropz,Dust2,Upper B,B anchor,2.01,0.57,118.0,51.1,0.1118,-0.0432,NaN,NaN
1491,IEM Cologne Major 2026,Major playoff,Vitality,Falcons,apEX,Dust2,Middle,CT,1.54,0.66,89.4,58.0,0.0588,-0.0453,NaN,NaN
1492,IEM Cologne Major 2026,Major playoff,Vitality,Falcons,ZywOo,Dust2,AWPer,AWPer,0.85,0.98,52.6,83.1,-0.0046,-0.0257,NaN,NaN
1493,IEM Cologne Major 2026,Major playoff,Vitality,Falcons,mezii,Dust2,Long,Middle,0.79,1.27,48.2,117.6,-0.0350,-0.0231,NaN,NaN
1494,IEM Cologne Major 2026,Major playoff,Vitality,Falcons,flameZ,Dust2,Lower B,Long,0.41,0.89,42.6,72.7,-0.0596,-0.0041,NaN,NaN
1495,PGL Astana 2025,Swiss Round,FURIA,The MongolZ,KSCERATO,Mirage,B extremity,A connector,1.93,1.43,102.8,80.0,0.0214,0.0617,NaN,NaN
1496,PGL Astana 2025,Swiss Round,FURIA,The MongolZ,FalleN,Mirage,Roamer,B anchor,1.26,0.37,74.8,31.2,0.0488,-0.0929,NaN,NaN
1497,PGL Astana 2025,Swiss Round,FURIA,The MongolZ,YEKINDAR,Mirage,Mid Brawl,B short,0.99,0.63,96.4,76.8,-0.1071,-0.0906,NaN,NaN
1498,PGL Astana 2025,Swiss Round,FURIA,The MongolZ,molodoy,Mirage,AWPer,AWPer,0.62,1.04,64.2,97.2,-0.0348,0.0082,NaN,NaN
1499,PGL Astana 2025,Swiss Round,FURIA,The MongolZ,yuurih,Mirage,A extremity,A anchor,0.46,0.43,48.8,42.8,-0.1275,-0.0961,NaN,NaN


In [112]:
import pandas as pd

df = pd.read_excel("CS2_Project.xlsx", sheet_name="Clean Data")

result = df.groupby("Player").agg({
    "T-side Rating":["mean","var", "std"],
    "CT-side Rating":["mean","var", "std"]
}).round(4)

print(result)

         T-side Rating                 CT-side Rating                
                  mean     var     std           mean     var     std
Player                                                               
FalleN          1.2600     NaN     NaN         0.3700     NaN     NaN
KSCERATO        1.9300     NaN     NaN         1.4300     NaN     NaN
YEKINDAR        0.9900     NaN     NaN         0.6300     NaN     NaN
ZywOo           1.4428  0.4870  0.6979         1.4229  0.3357  0.5794
apEX            0.9504  0.1830  0.4278         1.0883  0.2545  0.5044
flameZ          1.1757  0.2318  0.4814         1.2422  0.3773  0.6142
mezii           1.1132  0.2586  0.5085         1.1227  0.1764  0.4200
molodoy         0.6200     NaN     NaN         1.0400     NaN     NaN
ropz            1.2033  0.2694  0.5190         1.2181  0.2609  0.5108
yuurih          0.4600     NaN     NaN         0.4300     NaN     NaN


In [1]:
type(4.)

float